# Notebook 13 — Marketing Decision Layer

## Purpose

Convert the **final XGBoost disruption-risk predictions** from Notebook 12 into practical marketing actions.

This notebook does **not** retrain the model and does **not** use the test set to change model parameters.

The decision layer:
1. Loads the final test predictions.
2. Creates transparent risk bands from the model's disruption probability.
3. Maps risk bands to marketing actions.
4. Identifies the most relevant shipment-level risk factors available in the engineered data.
5. Produces a marketing decision table and summary.
6. Keeps the separate 2021 marketing campaign dataset separate from the 2024–2025 shipment test data; no invalid row-level merge is performed.
7. Does not claim that an action caused a particular ROI improvement.


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Project paths
ROOT = Path("..").resolve() if Path("..").exists() else Path(".").resolve()

DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models" / "supply_chain"
RESULTS_DIR = ROOT / "results"

PREDICTIONS_DIR = RESULTS_DIR / "predictions"
METRICS_DIR = RESULTS_DIR / "metrics"

for directory in [PREDICTIONS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Processed data exists:", DATA_DIR.exists())
print("Predictions directory exists:", PREDICTIONS_DIR.exists())


Project root: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Processed data exists: True
Predictions directory exists: True


In [4]:
# ============================================================
# Load Notebook 12 predictions and reconstruct the test rows
# ============================================================

PREDICTIONS_PATH = PREDICTIONS_DIR / "final_test_predictions.csv"

SHIPMENT_FEATURES_PATH = (
    DATA_DIR / "shipment_features_engineered.csv"
)

assert PREDICTIONS_PATH.exists(), (
    f"Missing Notebook 12 predictions: {PREDICTIONS_PATH}"
)

assert SHIPMENT_FEATURES_PATH.exists(), (
    f"Missing engineered shipment dataset: {SHIPMENT_FEATURES_PATH}"
)

# Load Notebook 12 predictions
predictions = pd.read_csv(PREDICTIONS_PATH)

# Load the engineered shipment dataset
shipment_data = pd.read_csv(SHIPMENT_FEATURES_PATH)

print("Predictions shape:", predictions.shape)
print("Shipment engineered data shape:", shipment_data.shape)

# ------------------------------------------------------------
# Validate Notebook 12 predictions
# ------------------------------------------------------------

assert len(predictions) == 750, (
    f"Expected 750 test predictions, found {len(predictions)}"
)

required_prediction_columns = [
    "actual_disruption",
    "predicted_disruption",
    "disruption_probability"
]

for col in required_prediction_columns:
    assert col in predictions.columns, (
        f"Missing prediction column: {col}"
    )

# ------------------------------------------------------------
# Recreate the exact chronological 70/15/15 split
# used in Notebook 07
# ------------------------------------------------------------

shipment_data["Date"] = pd.to_datetime(
    shipment_data["Date"],
    errors="coerce"
)

assert shipment_data["Date"].notna().all(), (
    "Shipment data contains invalid dates."
)

shipment_data = (
    shipment_data
    .sort_values("Date")
    .reset_index(drop=True)
)

total_rows = len(shipment_data)

train_end = int(total_rows * 0.70)
validation_end = train_end + int(total_rows * 0.15)

test_source = shipment_data.iloc[
    validation_end:
].copy().reset_index(drop=True)

print("Reconstructed test source shape:", test_source.shape)

# ------------------------------------------------------------
# Validate the reconstructed test set
# ------------------------------------------------------------

assert len(test_source) == 750, (
    f"Expected 750 reconstructed test rows, "
    f"found {len(test_source)}"
)

assert len(test_source) == len(predictions)

# ------------------------------------------------------------
# Align predictions with test shipment rows
# ------------------------------------------------------------

predictions = predictions.reset_index(drop=True)
test_source = test_source.reset_index(drop=True)

decision_df = test_source.copy()

decision_df["actual_disruption"] = (
    predictions["actual_disruption"].astype(int)
)

decision_df["predicted_disruption"] = (
    predictions["predicted_disruption"].astype(int)
)

decision_df["disruption_probability"] = (
    predictions["disruption_probability"].astype(float)
)

# ------------------------------------------------------------
# Final alignment validation
# ------------------------------------------------------------

assert len(decision_df) == 750

assert np.isfinite(
    decision_df["disruption_probability"]
).all()

assert (
    decision_df["disruption_probability"].between(0, 1).all()
)

print("\nPrediction columns:")
print(predictions.columns.tolist())

print("\nDecision table shape:", decision_df.shape)

print("\nDecision table preview:")

display(
    decision_df[
        [
            "Shipment_ID",
            "Date",
            "Origin_Port",
            "Destination_Port",
            "Transport_Mode",
            "Product_Category",
            "actual_disruption",
            "predicted_disruption",
            "disruption_probability"
        ]
    ].head(10)
)

print("\nTest source successfully reconstructed.")
print("Rows:", len(decision_df))
print("Features:", decision_df.shape[1])

Predictions shape: (750, 3)
Shipment engineered data shape: (5000, 51)
Reconstructed test source shape: (750, 51)

Prediction columns:
['actual_disruption', 'predicted_disruption', 'disruption_probability']

Decision table shape: (750, 54)

Decision table preview:


,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,actual_disruption,predicted_disruption,disruption_probability
0,SC-13331,2025-09-19,Los Angeles,Rotterdam,Sea,Electronics,1,1,0.725645
1,SC-10630,2025-09-19,Los Angeles,Shanghai,Rail,Perishables,1,0,0.277615
2,SC-12453,2025-09-19,Rotterdam,Marseille,Road,Perishables,0,0,0.154232
3,SC-14320,2025-09-20,Singapore,Busan,Road,Textiles,1,1,0.998052
4,SC-14104,2025-09-20,Dubai,Hamburg,Road,Electronics,0,0,0.235215
5,SC-11598,2025-09-20,Dubai,Busan,Rail,Electronics,1,1,0.787641
6,SC-14233,2025-09-20,Hamburg,Shanghai,Rail,Automotive,0,1,0.918017
7,SC-13144,2025-09-20,Antwerp,Rotterdam,Rail,Electronics,0,0,0.439224
8,SC-14197,2025-09-20,Busan,Dubai,Road,Pharmaceuticals,0,0,0.476650
9,SC-13414,2025-09-20,Singapore,Dubai,Air,Textiles,0,0,0.355589



Test source successfully reconstructed.
Rows: 750
Features: 54


In [5]:
# Reconstruct the shipment-level decision table

# Notebook 07's test.csv preserves the original row order.
# Notebook 12 predictions were generated in that same order.
# Therefore, rows can be aligned by position without inventing a new key.

decision_df = test_source.copy().reset_index(drop=True)
predictions = predictions.reset_index(drop=True)

decision_df["actual_disruption"] = predictions["actual_disruption"].astype(int)
decision_df["predicted_disruption"] = predictions["predicted_disruption"].astype(int)
decision_df["disruption_probability"] = predictions["disruption_probability"].astype(float)

assert len(decision_df) == 750
assert np.isfinite(decision_df["disruption_probability"]).all()

print("Decision table shape:", decision_df.shape)
display(
    decision_df[
        [
            "Shipment_ID",
            "Date",
            "Origin_Port",
            "Destination_Port",
            "Transport_Mode",
            "Product_Category",
            "predicted_disruption",
            "disruption_probability",
        ]
    ].head(10)
)


Decision table shape: (750, 54)


,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,predicted_disruption,disruption_probability
0,SC-13331,2025-09-19,Los Angeles,Rotterdam,Sea,Electronics,1,0.725645
1,SC-10630,2025-09-19,Los Angeles,Shanghai,Rail,Perishables,0,0.277615
2,SC-12453,2025-09-19,Rotterdam,Marseille,Road,Perishables,0,0.154232
3,SC-14320,2025-09-20,Singapore,Busan,Road,Textiles,1,0.998052
4,SC-14104,2025-09-20,Dubai,Hamburg,Road,Electronics,0,0.235215
5,SC-11598,2025-09-20,Dubai,Busan,Rail,Electronics,1,0.787641
6,SC-14233,2025-09-20,Hamburg,Shanghai,Rail,Automotive,1,0.918017
7,SC-13144,2025-09-20,Antwerp,Rotterdam,Rail,Electronics,0,0.439224
8,SC-14197,2025-09-20,Busan,Dubai,Road,Pharmaceuticals,0,0.476650
9,SC-13414,2025-09-20,Singapore,Dubai,Air,Textiles,0,0.355589


In [7]:
# Create transparent risk bands

# These thresholds are business decision rules, not model-trained thresholds.
# They are intentionally documented and are NOT optimized using the test set.

def risk_band(probability):
    if probability >= 0.75:
        return "High"
    elif probability >= 0.50:
        return "Medium"
    else:
        return "Low"

decision_df["risk_band"] = decision_df["disruption_probability"].apply(risk_band)

risk_order = ["Low", "Medium", "High"]
decision_df["risk_band"] = pd.Categorical(
    decision_df["risk_band"],
    categories=risk_order,
    ordered=True
)

print("Risk band counts:")
print(decision_df["risk_band"].value_counts().sort_index())


Risk band counts:
risk_band
Low       275
Medium    174
High      301
Name: count, dtype: int64


In [8]:
# Map disruption risk to marketing actions

# The actions are decision-layer recommendations.
# They are not claims about measured ROI or causal effectiveness.

MARKETING_ACTIONS = {
    "Low": {
        "marketing_action": "Continue planned campaigns",
        "budget_action": "Maintain planned spend",
        "message": "Normal campaign activity; no disruption-specific intervention."
    },
    "Medium": {
        "marketing_action": "Use disruption-aware targeting",
        "budget_action": "Prioritize flexible channels and monitor spend",
        "message": "Review campaign timing, audience exposure, and product availability."
    },
    "High": {
        "marketing_action": "Protect demand and reallocate exposure",
        "budget_action": "Reduce exposure to affected shipment-dependent products and prioritize resilient alternatives",
        "message": "Coordinate marketing with supply-chain risk before increasing demand."
    },
}

decision_df["marketing_action"] = decision_df["risk_band"].map(
    lambda x: MARKETING_ACTIONS[str(x)]["marketing_action"]
)

decision_df["budget_action"] = decision_df["risk_band"].map(
    lambda x: MARKETING_ACTIONS[str(x)]["budget_action"]
)

decision_df["decision_message"] = decision_df["risk_band"].map(
    lambda x: MARKETING_ACTIONS[str(x)]["message"]
)

display(
    decision_df[
        [
            "Shipment_ID",
            "risk_band",
            "disruption_probability",
            "marketing_action",
            "budget_action",
            "decision_message",
        ]
    ].head(15)
)


,Shipment_ID,risk_band,disruption_probability,marketing_action,budget_action,decision_message
0,SC-13331,Medium,0.725645,Use disruption-aware targeting,Prioritize flexible channels and monitor spend,"Review campaign timing, audience exposure, and..."
1,SC-10630,Low,0.277615,Continue planned campaigns,Maintain planned spend,Normal campaign activity; no disruption-specif...
2,SC-12453,Low,0.154232,Continue planned campaigns,Maintain planned spend,Normal campaign activity; no disruption-specif...
3,SC-14320,High,0.998052,Protect demand and reallocate exposure,Reduce exposure to affected shipment-dependent...,Coordinate marketing with supply-chain risk be...
4,SC-14104,Low,0.235215,Continue planned campaigns,Maintain planned spend,Normal campaign activity; no disruption-specif...
5,SC-11598,High,0.787641,Protect demand and reallocate exposure,Reduce exposure to affected shipment-dependent...,Coordinate marketing with supply-chain risk be...
6,SC-14233,High,0.918017,Protect demand and reallocate exposure,Reduce exposure to affected shipment-dependent...,Coordinate marketing with supply-chain risk be...
7,SC-13144,Low,0.439224,Continue planned campaigns,Maintain planned spend,Normal campaign activity; no disruption-specif...
8,SC-14197,Low,0.476650,Continue planned campaigns,Maintain planned spend,Normal campaign activity; no disruption-specif...
9,SC-13414,Low,0.355589,Continue planned campaigns,Maintain planned spend,Normal campaign activity; no disruption-specif...


In [9]:
# Identify transparent shipment risk factors

# These are observable engineered/source features.
# They are NOT SHAP explanations and are not claimed to be causal.

risk_factor_columns = [
    "Geopolitical_Risk_Score",
    "Weather_Condition",
    "Lead_Time_Days",
    "Carrier_Reliability_Score",
    "Fuel_Price_Index",
    "shipping_supply_chain_pressure_index",
    "shipping_on_time_delivery_pct",
    "commodity_price",
    "tariff_tariff_rate_pct",
    "disruption_event_flag",
]

available_risk_factors = [
    col for col in risk_factor_columns
    if col in decision_df.columns
]

print("Available risk-factor columns:")
for col in available_risk_factors:
    print("-", col)

assert len(available_risk_factors) > 0


Available risk-factor columns:
- Geopolitical_Risk_Score
- Weather_Condition
- Lead_Time_Days
- Carrier_Reliability_Score
- Fuel_Price_Index
- shipping_supply_chain_pressure_index
- shipping_on_time_delivery_pct
- commodity_price
- tariff_tariff_rate_pct
- disruption_event_flag


In [10]:
# Build a simple human-readable risk-factor summary

def build_factor_summary(row):
    factors = []

    if "Geopolitical_Risk_Score" in row.index:
        if pd.notna(row["Geopolitical_Risk_Score"]) and row["Geopolitical_Risk_Score"] >= 7:
            factors.append("high geopolitical risk")

    if "Weather_Condition" in row.index:
        if str(row["Weather_Condition"]).strip().lower() in {
            "fog", "storm", "hurricane", "rain"
        }:
            factors.append(f"adverse weather ({row['Weather_Condition']})")

    if "Lead_Time_Days" in row.index:
        if pd.notna(row["Lead_Time_Days"]) and row["Lead_Time_Days"] >= 20:
            factors.append("long lead time")

    if "Carrier_Reliability_Score" in row.index:
        if pd.notna(row["Carrier_Reliability_Score"]) and row["Carrier_Reliability_Score"] < 0.80:
            factors.append("lower carrier reliability")

    if "shipping_supply_chain_pressure_index" in row.index:
        if pd.notna(row["shipping_supply_chain_pressure_index"]) and row["shipping_supply_chain_pressure_index"] >= 1:
            factors.append("elevated shipping pressure")

    if "shipping_on_time_delivery_pct" in row.index:
        if pd.notna(row["shipping_on_time_delivery_pct"]) and row["shipping_on_time_delivery_pct"] < 90:
            factors.append("lower on-time delivery")

    if "disruption_event_flag" in row.index:
        if pd.notna(row["disruption_event_flag"]) and row["disruption_event_flag"] == 1:
            factors.append("historical disruption event flag")

    if not factors:
        return "No predefined high-risk factor flag"

    return "; ".join(factors)

decision_df["risk_factor_summary"] = decision_df.apply(
    build_factor_summary,
    axis=1
)

display(
    decision_df[
        [
            "Shipment_ID",
            "risk_band",
            "disruption_probability",
            "risk_factor_summary",
        ]
    ].head(15)
)


,Shipment_ID,risk_band,disruption_probability,risk_factor_summary
0,SC-13331,Medium,0.725645,No predefined high-risk factor flag
1,SC-10630,Low,0.277615,adverse weather (Storm); long lead time
2,SC-12453,Low,0.154232,adverse weather (Storm)
3,SC-14320,High,0.998052,adverse weather (Rain); lower carrier reliability
4,SC-14104,Low,0.235215,adverse weather (Storm); lower carrier reliabi...
5,SC-11598,High,0.787641,adverse weather (Hurricane)
6,SC-14233,High,0.918017,lower carrier reliability
7,SC-13144,Low,0.439224,adverse weather (Rain)
8,SC-14197,Low,0.476650,adverse weather (Fog); lower carrier reliability
9,SC-13414,Low,0.355589,high geopolitical risk; adverse weather (Fog);...


In [11]:
# Marketing decision summary by risk band

risk_summary = (
    decision_df
    .groupby("risk_band", observed=False)
    .agg(
        shipments=("Shipment_ID", "count"),
        average_disruption_probability=("disruption_probability", "mean"),
        predicted_disruptions=("predicted_disruption", "sum"),
    )
    .reset_index()
)

risk_summary["shipment_share_pct"] = (
    risk_summary["shipments"] / len(decision_df) * 100
)

risk_summary["marketing_action"] = risk_summary["risk_band"].map(
    lambda x: MARKETING_ACTIONS[str(x)]["marketing_action"]
)

risk_summary["budget_action"] = risk_summary["risk_band"].map(
    lambda x: MARKETING_ACTIONS[str(x)]["budget_action"]
)

display(risk_summary)


,risk_band,shipments,average_disruption_probability,predicted_disruptions,shipment_share_pct,marketing_action,budget_action
0,Low,275,0.311127,0,36.666667,Continue planned campaigns,Maintain planned spend
1,Medium,174,0.610643,174,23.200000,Use disruption-aware targeting,Prioritize flexible channels and monitor spend
2,High,301,0.942217,301,40.133333,Protect demand and reallocate exposure,Reduce exposure to affected shipment-dependent...


In [12]:
# Optional operational segmentation for marketing planning

# This is descriptive segmentation only.
# It does not estimate campaign ROI and does not merge the 2021 marketing data
# with the 2024–2025 shipment records.

segment_summary = (
    decision_df
    .groupby(
        ["risk_band", "Transport_Mode", "Product_Category"],
        observed=False
    )
    .agg(
        shipments=("Shipment_ID", "count"),
        average_risk=("disruption_probability", "mean"),
    )
    .reset_index()
    .sort_values(
        ["risk_band", "average_risk"],
        ascending=[True, False]
    )
)

display(segment_summary.head(25))


,risk_band,Transport_Mode,Product_Category,shipments,average_risk
9,Low,Rail,Textiles,12,0.378654
1,Low,Air,Electronics,10,0.377260
13,Low,Road,Pharmaceuticals,18,0.349657
3,Low,Air,Pharmaceuticals,18,0.343401
10,Low,Road,Automotive,11,0.330993
19,Low,Sea,Textiles,12,0.324999
14,Low,Road,Textiles,15,0.323070
16,Low,Sea,Electronics,11,0.322819
18,Low,Sea,Pharmaceuticals,11,0.321142
8,Low,Rail,Pharmaceuticals,11,0.312279


In [13]:
# Save the complete marketing decision layer

DECISION_PATH = PREDICTIONS_DIR / "marketing_decision_layer.csv"
RISK_SUMMARY_PATH = METRICS_DIR / "marketing_risk_summary.csv"
SEGMENT_PATH = METRICS_DIR / "marketing_risk_segment_summary.csv"

decision_df.to_csv(DECISION_PATH, index=False)
risk_summary.to_csv(RISK_SUMMARY_PATH, index=False)
segment_summary.to_csv(SEGMENT_PATH, index=False)

print("Saved:")
print("-", DECISION_PATH)
print("-", RISK_SUMMARY_PATH)
print("-", SEGMENT_PATH)


Saved:
- C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\results\predictions\marketing_decision_layer.csv
- C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\results\metrics\marketing_risk_summary.csv
- C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\results\metrics\marketing_risk_segment_summary.csv


In [14]:
# Save the documented decision rules

decision_rules = {
    "purpose": "Map final disruption probability to marketing actions.",
    "source_model": "final_xgboost_candidate.joblib",
    "probability_source": "Notebook 12 final_test_predictions.csv",
    "risk_bands": {
        "Low": {
            "condition": "probability < 0.50",
            "marketing_action": MARKETING_ACTIONS["Low"]["marketing_action"],
            "budget_action": MARKETING_ACTIONS["Low"]["budget_action"],
        },
        "Medium": {
            "condition": "0.50 <= probability < 0.75",
            "marketing_action": MARKETING_ACTIONS["Medium"]["marketing_action"],
            "budget_action": MARKETING_ACTIONS["Medium"]["budget_action"],
        },
        "High": {
            "condition": "probability >= 0.75",
            "marketing_action": MARKETING_ACTIONS["High"]["marketing_action"],
            "budget_action": MARKETING_ACTIONS["High"]["budget_action"],
        },
    },
    "test_used_for_model_training": False,
    "test_used_for_hyperparameter_tuning": False,
    "test_used_for_model_selection": False,
    "test_used_to_optimize_risk_band_thresholds": False,
    "marketing_2021_data_row_level_merged_with_test": False,
    "causal_roi_claim_made": False,
}

RULES_PATH = MODEL_DIR / "marketing_decision_rules.json"
RULES_PATH.write_text(
    json.dumps(decision_rules, indent=2),
    encoding="utf-8"
)

print("Saved:", RULES_PATH)


Saved: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\marketing_decision_rules.json


In [17]:
# ============================================================
# Notebook 13 — Final Validation — CORRECTED
# ============================================================

# ------------------------------------------------------------
# Positive checks — these must be TRUE
# ------------------------------------------------------------

positive_checks = {

    # Notebook 12 prediction file
    "predictions_file_loaded": bool(
        PREDICTIONS_PATH.exists()
    ),

    # Actual source used to reconstruct the 750 test shipment rows
    "shipment_features_source_loaded": bool(
        SHIPMENT_FEATURES_PATH.exists()
    ),

    # Reconstructed decision table
    "decision_rows": bool(
        len(decision_df) == 750
    ),

    "probability_column_present": bool(
        "disruption_probability" in decision_df.columns
    ),

    "risk_band_present": bool(
        "risk_band" in decision_df.columns
    ),

    "marketing_action_present": bool(
        "marketing_action" in decision_df.columns
    ),

    "budget_action_present": bool(
        "budget_action" in decision_df.columns
    ),

    "risk_factor_summary_present": bool(
        "risk_factor_summary" in decision_df.columns
    ),

    "risk_bands_valid": bool(
        decision_df["risk_band"]
        .isin(["Low", "Medium", "High"])
        .all()
    ),

    "probabilities_valid": bool(
        np.isfinite(
            decision_df["disruption_probability"]
        ).all()
        and (
            decision_df["disruption_probability"] >= 0
        ).all()
        and (
            decision_df["disruption_probability"] <= 1
        ).all()
    ),

    # Output files
    "decision_file_exists": bool(
        DECISION_PATH.exists()
    ),

    "risk_summary_exists": bool(
        RISK_SUMMARY_PATH.exists()
    ),

    "segment_summary_exists": bool(
        SEGMENT_PATH.exists()
    ),

    "decision_rules_exists": bool(
        RULES_PATH.exists()
    ),
}


# ------------------------------------------------------------
# Integrity / leakage checks
# These MUST be FALSE
# ------------------------------------------------------------

leakage_checks = {

    "test_used_for_model_training": False,

    "test_used_for_hyperparameter_tuning": False,

    "test_used_for_model_selection": False,

    "test_used_to_optimize_risk_band_thresholds": False,

    "marketing_2021_data_row_level_merged_with_test": False,

    "causal_roi_claim_made": False,
}


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 70)
print("NOTEBOOK 13 — MARKETING DECISION LAYER VALIDATION")
print("=" * 70)

print("\nPositive checks — expected TRUE:")
print("-" * 70)

for key, value in positive_checks.items():
    print(f"{key}: {value}")


print("\nIntegrity / leakage checks — expected FALSE:")
print("-" * 70)

for key, value in leakage_checks.items():
    print(f"{key}: {value}")


# ------------------------------------------------------------
# Identify failures
# ------------------------------------------------------------

failed_positive = [
    key
    for key, value in positive_checks.items()
    if value is not True
]

failed_leakage = [
    key
    for key, value in leakage_checks.items()
    if value is not False
]


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)

if not failed_positive and not failed_leakage:

    print("All positive checks passed.")
    print("All integrity/leakage checks passed.")

    print("=" * 70)

    print("\nNotebook 13 final validation: PASS")

    print(
        "Marketing decision layer generated successfully."
    )

else:

    if failed_positive:
        print("FAILED POSITIVE CHECKS:")
        for key in failed_positive:
            print(f"- {key}")

    if failed_leakage:
        print("\nFAILED INTEGRITY/LEAKAGE CHECKS:")
        for key in failed_leakage:
            print(f"- {key}")

    print("=" * 70)

    # Print the exact problem instead of hiding it
    print("\nNotebook 13 final validation: FAIL")

    print("\nPositive failures:", failed_positive)
    print("Leakage failures:", failed_leakage)

    raise AssertionError(
        "Notebook 13 validation failed. "
        "See the failed checks printed above."
    )

NOTEBOOK 13 — MARKETING DECISION LAYER VALIDATION

Positive checks — expected TRUE:
----------------------------------------------------------------------
predictions_file_loaded: True
shipment_features_source_loaded: True
decision_rows: True
probability_column_present: True
risk_band_present: True
marketing_action_present: True
budget_action_present: True
risk_factor_summary_present: True
risk_bands_valid: True
probabilities_valid: True
decision_file_exists: True
risk_summary_exists: True
segment_summary_exists: True
decision_rules_exists: True

Integrity / leakage checks — expected FALSE:
----------------------------------------------------------------------
test_used_for_model_training: False
test_used_for_hyperparameter_tuning: False
test_used_for_model_selection: False
test_used_to_optimize_risk_band_thresholds: False
marketing_2021_data_row_level_merged_with_test: False
causal_roi_claim_made: False

All positive checks passed.
All integrity/leakage checks passed.

Notebook 13 fin